In [ ]:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%%capture
!pip install unsloth
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
import unsloth

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
from unsloth import FastLanguageModel

In [ ]:
import torch
import pandas as pd
import torch.nn.functional as F

In [ ]:
!pip install flash-attn

In [ ]:
import numpy as np
from tqdm.auto import tqdm

In [ ]:
import re
import string
import torch
import pandas as pd
from functools import lru_cache
from collections import defaultdict

In [ ]:
def install_llama():
  max_seq_length = 2048
  dtype = None
  load_in_4bit = True


  fourbit_models = [
      "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
      "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
      "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
      "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
      "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
      "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
      "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
      "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
      "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
      "unsloth/Phi-3-medium-4k-instruct",
      "unsloth/gemma-2-9b-bnb-4bit",
      "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
      ]

  model, tokenizer = FastLanguageModel.from_pretrained(
      model_name = "unsloth/Meta-Llama-3.1-8B",
      max_seq_length = max_seq_length,
      dtype = dtype,
      load_in_4bit = True
      )
  return model, tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
model, tokenizer = install_llama()

In [ ]:
cyrillic_re = re.compile(r'[\u0400-\u04FF]')

def is_cyrillic(text):
    return bool(re.search(r'[а-яёА-ЯЁ]', text))

In [ ]:
SEPARATORS = set([' ', ',', '.', ';', ':', '!', '?', '\n', '\t'])

In [ ]:
def is_valid_token(token_str):
    token_str = token_str.strip()
    if is_cyrillic(token_str):
        return True
    if token_str in ["", " ", "\n", "\t"]:
        return True
    if re.match(r'^[\.,!?-]+$', token_str):
        return True
    return False

In [ ]:
def is_space_token(token_str):
    """Проверяет, является ли токен пробелом"""
    return token_str.strip() == "" and token_str != ""


In [ ]:
def ends_with_separator(token_str):
    """Проверяет, заканчивается ли токен на разделитель или пунктуацию"""
    return bool(re.search(r'[ .,!?;:"\']$', token_str))


In [ ]:
def extract_separator(token_str):
  """
  Функция извлекает сепаратор из токена.
  """
  if token_str and token_str[0] in SEPARATORS:
      return token_str[0]
  if token_str and token_str[-1] in SEPARATORS:
      return token_str[-1]
  return None

In [ ]:
def contains_separator(token_str):
    return (token_str and (token_str[0] in SEPARATORS or token_str[-1] in SEPARATORS))


In [ ]:
def is_single_cyrillic_word(word):
    if not word or not word.strip():
        return False
    word = word.strip()
    return bool(re.match(r'^[а-яёА-ЯЁ-]+$', word))

In [ ]:
def initialize_token_structures(tokenizer):
    separator_strings = [" ", ",", ".", ";", ":", "!", "?", "\n", "\t"]
    separator_ids = set()

    for sep in separator_strings:
        tokens = tokenizer.encode(sep, add_special_tokens=False)
        separator_ids.update(tokens)

    special_tokens = []
    if tokenizer.eos_token_id is not None:
        special_tokens.append(tokenizer.eos_token_id)
    if tokenizer.bos_token_id is not None:
        special_tokens.append(tokenizer.bos_token_id)
    if tokenizer.pad_token_id is not None:
        special_tokens.append(tokenizer.pad_token_id)
    if tokenizer.unk_token_id is not None:
        special_tokens.append(tokenizer.unk_token_id)

    separator_ids.update(special_tokens)

    original_vocab_size = tokenizer.vocab_size
    cyrillic_token_ids = set()

    for token_id in range(original_vocab_size):
        try:
            token_str = tokenizer.decode([token_id], skip_special_tokens=False)
            if cyrillic_re.search(token_str.strip()) or token_id in separator_ids:
                cyrillic_token_ids.add(token_id)
        except Exception:
            pass

    cyrillic_token_ids.update(special_tokens)
    cyrillic_token_ids = sorted(cyrillic_token_ids)

    new_id_to_old_id = {new_id: old_id for new_id, old_id in enumerate(cyrillic_token_ids)}
    old_id_to_new_id = {old_id: new_id for new_id, old_id in enumerate(cyrillic_token_ids)}

    print(f"Reduced vocabulary size: {len(cyrillic_token_ids)} / {original_vocab_size}")

    token_cache = {}
    for token_id in cyrillic_token_ids:
        token_str = tokenizer.decode([token_id])
        token_cache[token_id] = {
            'str': token_str,
            'is_cyrillic': is_cyrillic(token_str.strip()),
            'is_valid': is_valid_token(token_str.strip()),
            'is_separator': token_str.strip() in separator_strings or token_str.strip() in string.punctuation,
            'is_space': is_space_token(token_str),
            'ends_with_separator': ends_with_separator(token_str),
            'contains_separator': contains_separator(token_str)
        }

    return cyrillic_token_ids, new_id_to_old_id, old_id_to_new_id, token_cache

In [ ]:
@lru_cache(maxsize=2000)
def get_next_token_probs(input_ids_tuple, cyrillic_token_ids_tuple, model, device):
    input_ids = torch.tensor(input_ids_tuple, device=device).unsqueeze(0)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, return_dict=True)
        logits = outputs.logits[:, -1, :]
        reduced_logits = logits[:, list(cyrillic_token_ids_tuple)]
        probs = torch.softmax(reduced_logits, dim=-1).squeeze()
        return probs.cpu().numpy()

In [ ]:
def beam_search_completions3(sentence, tokenizer, model,
                             max_depth=10, beam_width=100, min_prob=1e-6,
                             cyrillic_token_ids=None, new_id_to_old_id=None,
                             token_cache=None, device=None):
    """
    Beam search с учётом сепараторов и мультитокеновых слов.
    Условия:
    1) Если первый токен начинается с сепаратора — убираем сепаратор и продолжаем генерацию.
    2) Если не первый токен и начинается с сепаратора — сохраняем текущее слово и прекращаем генерацию.
    3) Если токен — только сепаратор — прекращаем генерацию.
    4) Если сепаратор в конце токена — буквенную часть добавляем к слову и прекращаем генерацию.
    """
    if "<mask>" not in sentence:
        return None

    prefix, suffix = sentence.split("<mask>", 1)
    inputs = tokenizer(prefix.strip(), return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_ids = inputs["input_ids"]

    cyrillic_token_ids_tuple = tuple(cyrillic_token_ids)
    completed_words, word_details = {}, {}

    def save_word(tokens, probs, prob, depth, stop_reason, sep_prob=None):
        """Helper: save completed word if valid and better than existing one."""
        word_str = ''.join([token_cache[t]['str'] for t in tokens])
        if is_single_cyrillic_word(word_str):
            if word_str not in completed_words or prob > completed_words[word_str]:
                completed_words[word_str] = prob
                word_details[word_str] = {
                    'tokens': tokens.copy(),
                    'probs': probs.copy(),
                    'depth': depth,
                    'stop_reason': stop_reason,
                    'separator_probability': sep_prob
                }

    beams = [(input_ids, [], [], 1.0, 0)]  # (ids, tokens, probs, seq_prob, depth)

    for depth in range(max_depth):
        if not beams:
            break

        new_beams = []
        for ids, word, word_probs, seq_prob, d in beams:
            next_probs = get_next_token_probs(tuple(ids.squeeze().tolist()),
                                              cyrillic_token_ids_tuple, model, device)
            for reduced_id in np.argsort(next_probs)[-beam_width:][::-1]:
                prob = next_probs[reduced_id]
                total_prob = seq_prob * prob
                if total_prob < min_prob:
                    continue

                token_id = new_id_to_old_id[reduced_id]
                token_info = token_cache[token_id]
                if not token_info['is_valid']:
                    continue

                token_str = token_info['str']
                new_ids = torch.cat([ids, torch.tensor([[token_id]], device=device)], dim=1)

                sep_start = token_str[0] if token_str and token_str[0] in SEPARATORS else None
                sep_end = token_str[-1] if token_str and token_str[-1] in SEPARATORS and len(token_str) > 1 else None
                is_only_sep = token_str and all(c in SEPARATORS for c in token_str)

                # --- Rule handling ---
                if not word:  # 1) first token
                    if sep_start:
                        remainder = token_str[1:]
                        if remainder:
                            for tid in cyrillic_token_ids:
                                if token_cache[tid]['str'] == remainder:
                                    new_beams.append((new_ids, [tid], [prob], total_prob, d + 1))
                                    break
                    else:
                        new_beams.append((new_ids, [token_id], [prob], total_prob, d + 1))
                    continue

                if sep_start and not is_only_sep:  # 2) non-first, starts with sep
                    save_word(word, word_probs, seq_prob, d, {sep_start}, prob)
                    continue

                if is_only_sep:  # 3) only separator
                    if word:
                        save_word(word, word_probs, seq_prob, d, {token_str}, prob)
                    continue

                if sep_end:  # 4) ends with separator
                    part = token_str[:-1]
                    if part:
                        for tid in cyrillic_token_ids:
                            if token_cache[tid]['str'] == part:
                                new_word = word + [tid]
                                new_probs = word_probs + [prob]
                                save_word(new_word, new_probs, total_prob, d + 1, {sep_end}, prob)
                                break
                    continue

                # 5) regular continuation
                new_beams.append((new_ids, word + [token_id], word_probs + [prob], total_prob, d + 1))

        beams = sorted(new_beams, key=lambda x: x[3], reverse=True)[:beam_width]

    # Save unfinished words
    for _, word, probs, seq_prob, d in beams:
        if word:
            save_word(word, probs, seq_prob, d, "достигнута максимальная глубина")

    results = [{
        'word': w,
        'probability': completed_words[w],
        'token_probabilities': word_details[w]['probs'],
        'token_count': len(word_details[w]['tokens']),
        'depth': word_details[w]['depth'],
        'tokens': word_details[w]['tokens'],
        'token_texts': [token_cache[t]['str'] for t in word_details[w]['tokens']],
        'stop_reason': word_details[w]['stop_reason'],
        'separator_probability': word_details[w]['separator_probability']
    } for w in completed_words]

    return sorted(results, key=lambda x: x['probability'], reverse=True)


In [ ]:
from datetime import datetime

def create_complete_probability_table3(sentences, tokenizer, model,
                                      max_depth=5, beam_width=50, min_prob=1e-4,
                                      output_path=None):
    """
    Создает таблицу с вероятностями полных слов и сохраняет результаты в CSV-файл.

    Для долгосрочного хранения результатов рекомендуется указать путь к папке на Google Диске.

    Позже можно будет объединить все CSV-файлы из этой папки в одну таблицу для анализа.
    """
    device = model.device
    cyrillic_token_ids, new_id_to_old_id, old_id_to_new_id, token_cache = initialize_token_structures(tokenizer)
    all_results = []
    for sentence_idx, sentence in enumerate(sentences, 1):
        print(f"Обработка предложения {sentence_idx}: {sentence}")
        completions = beam_search_completions3(
            sentence, tokenizer, model,
            max_depth=max_depth,
            beam_width=beam_width,
            min_prob=min_prob,
            cyrillic_token_ids=cyrillic_token_ids,
            new_id_to_old_id=new_id_to_old_id,
            token_cache=token_cache,
            device=device
        )
        if completions:
            for completion in completions:
                completion['sentence_id'] = sentence_idx
                completion['sentence_text'] = sentence
                completion.setdefault('separator_probability', None)
                all_results.append(completion)
            print(f"  Найдено {len(completions)} вариантов")
        else:
            print(f"  Варианты не найдены")
    if all_results:
        df = pd.DataFrame(all_results)
        columns = ['sentence_id', 'sentence_text', 'word', 'probability',
                  'token_probabilities', 'token_count', 'depth', 'tokens',
                  'token_texts', 'stop_reason', 'separator_probability']
        df = df[columns]
        if output_path is None:
            output_path = f'lookup.csv'
        df.to_csv(output_path, index=False, encoding='utf-8')
        print(f"Результаты сохранены в: {output_path}")
        print(f"Всего вариантов: {len(df)}")
        return df
    else:
        print("Нет результатов для сохранения.")
        return None

In [ ]:
from google.colab import drive
import os

In [ ]:
import time
import threading
from IPython.display import display, Javascript

class ColabKeeper:
    def __init__(self):
        self.keep_running = True

    def js_keep_alive(self):
        display(Javascript('''
        function ClickConnect(){
            console.log("Keeping Colab alive");
            document.querySelector("colab-connect-button").click()
        }
        setInterval(ClickConnect, 55000)
        '''))

    def python_keep_alive(self):
        while self.keep_running:
            time.sleep(30)
            print(f"🔄 Activity: {time.ctime()}")

    def start(self):
        self.js_keep_alive()
        thread = threading.Thread(target=self.python_keep_alive, daemon=True)
        thread.start()

    def stop(self):
        self.keep_running = False

keeper = ColabKeeper()
keeper.start()

<IPython.core.display.Javascript object>

In [ ]:
from datetime import datetime

In [ ]:
left_contexts = ["В тот момент <mask>",
"Клиенты воровали из ресторана <mask>",
"Ему удалось вскрыть банку об острый край <mask>",
"Ей никак не суметь <mask>",
"Старуха была страшной -- <mask>",
"Выбирая вязаную шапочку, знайте, что лучше шапка цвета <mask>",
"У моего отца был счёт в швейцарском <mask>",
"Очень хочется заплести <mask>",
"В котёл бросают куски <mask>",
"На запись голоса <mask>",
"Ее сын Гриша умер <mask>",
"Можно будет <mask>",
"Музыканты играли на похоронах, разгружали <mask>",
"И не надо ставить это целью <mask>",
"Дрозды и сковрцы начали <mask>",
"Что может сделать самый сильный <mask>",
"Здесь потребуется <mask>",
"Какие главные лекарства должны <mask>",
"Применение микросхемы <mask>",
"Выходя замуж, ты надеялась обрести спокойствие, уютный <mask>",
"С нескрываемой <mask>",
"Однако здесь <mask>",
"Тому, кто <mask>",
"В качестве примера приводится <mask>",
"Он признаёт право каждого <mask>",
"Вспоминая <mask>",
"Он вскрыл пачку сухарей, <mask>",
"Но четыре года я не мог себя <mask>",
"На Ольгу Васильевну было написано <mask>",
"Я знал: их особенно <mask>",
"Очень тогда <mask>",
"Создать настоящие шедевры вам помогут <mask>",
"Наши власти позволяют себе <mask>",
"У нас в Волгограде многие придерживаются <mask>",
"Во избежание ожогов надо нанести на лицо небольшое <mask>",
"В вопросе послышался упрёк <mask>",
"За углом ― Морской музей, с бесчисленными моделями <mask>",
"Мне нравится сын коллеги, <mask>",
"Душа требовала <mask>",
"Наше правительство сделало <mask>",
"А промывать манную <mask>",
"Каждое утро на самый верх <mask>",
"Убедительно просим вас разборчиво заполнять <mask>",
"На болотах оставался ещё <mask>",
"Дорога ведет в глухой <mask>",
"На ведущей вниз <mask>",
"Когда она в самолёте <mask>",
"Возможности этих перемен будут обсуждаться в Париже <mask>",
"Там, недалеко от кухонной двери, сидел <mask>",
"В багажнике были лопата, <mask>",
"Врач прописал заживляющую <mask>",
"В бассейне <mask>",
"В резервациях <mask>",
"Мама брала меня с собой, и мы, сдав <mask>",
"Приблизительно в центре тайги <mask>",
"Не поручайте <mask>",
"В деревнях по-прежнему <mask>",
"В числе возможных кандидатов <mask>",
"Твоё тело расслабляется, и исчезает <mask>",
"Он был очень <mask>",
"Они не ели целый день, <mask>",
"В речи учёного прозвучало <mask>",
"Тема эта в то время была <mask>",
"Существует легенда, что <mask>",
"Он ловко поддел концом <mask>",
"У директора школы был тонкий <mask>",
"У Пашки <mask>",
"В конверте вместе с деньгами была <mask>",
"Стала стабильнее экономическая и политическая <mask>",
"В современном <mask>",
"Педагог предъявляет <mask>",
"Я сказал, что русский солдат <mask>",
"Старый шкаф <mask>",
"В сюжете этого фильма какие-то <mask>",
"Ирине досталась <mask>",
"На газовой плите стояла <mask>",
"Я люблю салат из картошки с зеленью, заправленный <mask>",
"Ненужный коврик из твёрдой <mask>",
"Что ты хочешь чтобы тебе <mask>",
"Журналист взял карандаш, <mask>",
"У мамы есть <mask>",
"Отвернув цветастое <mask>",
"У тебя впереди замечательный день, <mask>",
"Этот студент <mask>",
"Она почти не изменилась, только слегка <mask>",
"Перед ним снова была <mask>",
"Торговля продуктами питания является одной из самых <mask>",
"Когда родители <mask>",
"Товарищ генерал, <mask>",
"Считается, что коллекционирование <mask>",
"Телята быстро <mask>",
"Один футболист, который получил <mask>",
"Судя по огромному <mask>",
"Город, раскинувшийся вдоль <mask>",
"Под рукавом рубашки виднелся тонкий <mask>",
"На вторичном рынке жилья <mask>",
"Ваня раскрыл было <mask>",
"В мои обязанности входило утром включить <mask>",
"И на берегу озера тогда появляются <mask>",
"Наживка, на которую он ловил <mask>",
"Не обнаружив ничего в досье, сыщики решили <mask>",
"Когда мне хотелось <mask>",
"Взяв с собой фотоаппарат, вся <mask>",
"Собаку, виновницу случившегося, приказали <mask>",
"Мне было лень идти на стоянку и сметать <mask>",
"От смерти его спасла <mask>",
"По воскресеньям музыканты, исполнявшие <mask>",
"Он умел из любого <mask>",
"Если я еще увижу здесь хоть <mask>",
"Количество денег в обороте выросло благодаря <mask>",
"Он стал плохо <mask>",
"Работы выполняет <mask>",
"Зачем ему звонить, если откликается <mask>",
"Я слезал, щупал <mask>",
"Шею Лизы украшало ожерелье <mask>",
"Этот роман захватывает читателя с первой <mask>",
"Автор принадлежит к числу последних свидетелей <mask>",
"В темноте Иван задел острый <mask>",
"Приготовь себе диетические овощные блюда, <mask>",
"Олень бродил среди берёз, жевал <mask>",
"Причиной аварии был мобильный <mask>",
"После завершения <mask>",
"Чтобы придать объем <mask>",
"В лесу ветром <mask>",
"В каждом <mask>",
"Власть судов была такой <mask>",
"Володя каким-то образом <mask>",
"За два года накопилась <mask>",
"Если мы позволим этим людям <mask>",
"Думаю, большой <mask>",
"Елена сидела в кресле, молодая Мурка <mask>",
"От внимания наблюдателя не должна <mask>",
"Государством предлагается <mask>",
"Сделав мне знак помолчать, он приложил <mask>",
"Она успевала убраться, разморозить <mask>",
"Что используют для этой прически, <mask>",
"Она с досадой <mask>",
"Зоопарк ― это кусочек другого мира, находящийся в самом <mask>",
"Я сделала <mask>",
"За министром труда тянется целый <mask>",
"Мы установили камеру на новый <mask>",
"Ей хотелось выплеснуть чай на бежевый <mask>",
"На привале у озера <mask>",
"Покуда я нахожусь у власти, я буду предметом <mask>"]

In [ ]:
%%time

if __name__ == "__main__":

  for context in left_contexts:

    probability_table = create_complete_probability_table3(
          sentences=[context],
          tokenizer=tokenizer,
          model=model,
          max_depth=20,
          beam_width=200,
          min_prob=1e-6
    )
